# 📊 EduTrack Analytics — Notebook d'Analyse Exploratoire
**MAROC YNOV CAMPUS — Projet Spé DATA 2026**

Dataset : 300 étudiants · 9 000 notes · 10 modules · 3 semestres · 6 classes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')
plt.style.use('dark_background')
sns.set_palette('husl')
print('✅ Librairies chargées')

## 1. Chargement des Données

In [ ]:
df_etu   = pd.read_csv('data/etudiants.csv')
df_notes = pd.read_csv('data/notes.csv')
df_abs   = pd.read_csv('data/absences.csv')
print(f'Étudiants : {len(df_etu):,} lignes, {df_etu.shape[1]} colonnes')
print(f'Notes     : {len(df_notes):,} lignes, {df_notes.shape[1]} colonnes')
print(f'Absences  : {len(df_abs):,} lignes, {df_abs.shape[1]} colonnes')
print(f'Classes   : {df_etu["classe"].nunique()} promotions')
print(f'Modules   : {df_notes["module"].nunique()} matières')
df_etu.head(3)

## 2. Nettoyage & Validation

In [ ]:
print('=== Valeurs manquantes ==='); print(df_notes.isnull().sum())
print(f'\n=== Doublons ===' ); print(f'Notes : {df_notes.duplicated().sum()}')
print('\n=== Statistiques notes ==='); print(df_notes['note'].describe().round(2))
df_notes['note'] = pd.to_numeric(df_notes['note'], errors='coerce').clip(0, 20)
df_notes = df_notes.drop_duplicates()
print('\n✅ Données nettoyées et validées')

## 3. Statistiques Descriptives par Module

In [ ]:
stats = df_notes.groupby('module')['note'].agg(
    Moyenne='mean', Mediane='median', Ecart_type='std',
    Min='min', Max='max',
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75),
    Taux_reussite=lambda x: (x>=10).mean()*100
).round(2)
stats.sort_values('Moyenne', ascending=False)

## 4. Analyse par Semestre (Progression)

In [ ]:
prog = df_notes.groupby('semestre').agg(
    moyenne=('note','mean'),
    taux_reussite=('note', lambda x: (x>=10).mean()*100)
).round(2)
print(prog)
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(prog.index, prog['moyenne'], 'o-', color='#4f8ef7', lw=2.5, label='Moyenne/20')
ax2=ax.twinx()
ax2.plot(prog.index, prog['taux_reussite'], 's--', color='#10b981', lw=2.5, label='Réussite %')
ax.set_title('Progression Semestre par Semestre')
ax.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.tight_layout(); plt.savefig('progression_semestres.png',dpi=150); plt.show()

## 5. Visualisations EDA (6 graphiques)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18,10))
fig.suptitle('EduTrack Analytics — Analyse Exploratoire (300 étudiants)', fontsize=15, fontweight='bold')

# 1. Distribution globale
axes[0,0].hist(df_notes['note'],bins=20,color='#4f8ef7',edgecolor='white',alpha=0.8)
axes[0,0].axvline(10,color='red',linestyle='--',label='Seuil 10')
axes[0,0].set_title('Distribution des Notes'); axes[0,0].legend()

# 2. Moyenne par module
moy=df_notes.groupby('module')['note'].mean().sort_values()
colors=['#f05252' if v<10 else '#10b981' for v in moy.values]
axes[0,1].barh(moy.index,moy.values,color=colors,alpha=0.85)
axes[0,1].axvline(10,color='red',linestyle='--')
axes[0,1].set_title('Moyenne par Module'); axes[0,1].set_xlabel('Moyenne /20')

# 3. Distribution par classe
merged=df_notes.merge(df_etu[['etudiant_id','classe']],on='etudiant_id')
for c in merged['classe'].unique():
    axes[0,2].hist(merged[merged['classe']==c]['note'],bins=12,alpha=0.55,label=c)
axes[0,2].set_title('Distribution par Classe'); axes[0,2].legend(fontsize=7)

# 4. Corrélation absences / moyenne
moy_etu=df_notes.groupby('etudiant_id')['note'].mean()
abs_etu=df_abs.groupby('etudiant_id').size()
cdf=pd.DataFrame({'moyenne':moy_etu,'absences':abs_etu}).dropna()
axes[1,0].scatter(cdf['absences'],cdf['moyenne'],alpha=0.4,color='#7b6ff0')
axes[1,0].set_title(f'Absences vs Moy (r={cdf.corr().iloc[0,1]:.3f})')
axes[1,0].set_xlabel('Nb Absences'); axes[1,0].set_ylabel('Moyenne /20')

# 5. Taux de réussite
taux=df_notes.groupby('module')['note'].apply(lambda x:(x>=10).mean()*100).sort_values()
colors2=['#f05252' if t<50 else '#f59e0b' if t<70 else '#10b981' for t in taux]
axes[1,1].barh(taux.index,taux.values,color=colors2,alpha=0.85)
axes[1,1].set_title('Taux de Réussite par Module'); axes[1,1].set_xlabel('%')

# 6. Segments
moyennes=df_notes.groupby('etudiant_id')['note'].mean()
seg=pd.cut(moyennes,bins=[0,8,10,12,15,20],labels=['Critique','Fragile','Moyen','Bon','Excellent'])
seg_c=seg.value_counts()
axes[1,2].pie(seg_c.values,labels=seg_c.index,autopct='%1.1f%%',colors=['#f05252','#f59e0b','#7b6ff0','#4f8ef7','#10b981'])
axes[1,2].set_title('Segmentation des Étudiants')

plt.tight_layout()
plt.savefig('analyse_exploratoire.png',dpi=150,bbox_inches='tight'); plt.show()
print('✅ 6 visualisations générées')

## 6. Bonus A — Modèle Prédictif (Random Forest)

In [ ]:
features = pd.DataFrame({
    'moyenne':     df_notes.groupby('etudiant_id')['note'].mean(),
    'ecart_type':  df_notes.groupby('etudiant_id')['note'].std().fillna(0),
    'nb_modules':  df_notes.groupby('etudiant_id')['module'].nunique(),
    'nb_absences': df_abs.groupby('etudiant_id').size(),
}).fillna(0)
y = (features['moyenne'] < 10).astype(int)
X = features
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
model = RandomForestClassifier(n_estimators=100,random_state=42,max_depth=6)
model.fit(X_train_s,y_train)
y_pred = model.predict(X_test_s)
print('=== Rapport de Classification ===')
print(classification_report(y_test,y_pred,target_names=['Non-Risque','À Risque']))
fig,axes=plt.subplots(1,2,figsize=(12,4))
ConfusionMatrixDisplay.from_predictions(y_test,y_pred,display_labels=['Non-Risque','À Risque'],ax=axes[0],colorbar=False,cmap='Blues')
axes[0].set_title('Matrice de Confusion')
importances=pd.Series(model.feature_importances_,index=X.columns).sort_values()
importances.plot(kind='barh',ax=axes[1],color='#4f8ef7')
axes[1].set_title('Importance des Variables')
plt.tight_layout(); plt.savefig('ml_risk_model.png',dpi=150); plt.show()

## 7. Bonus B — Clustering KMeans

In [ ]:
X_c = features[['moyenne','nb_absences','ecart_type']].fillna(0)
X_s = StandardScaler().fit_transform(X_c)
kmeans = KMeans(n_clusters=4,random_state=42,n_init=10)
features['cluster'] = kmeans.fit_predict(X_s)
print('=== Profils des 4 Clusters ===')
sum_c = features.groupby('cluster').agg(moyenne=('moyenne','mean'),absences=('nb_absences','mean'),nb=('cluster','count')).round(2)
print(sum_c)
fig,ax=plt.subplots(figsize=(10,6))
sc=ax.scatter(features['nb_absences'],features['moyenne'],c=features['cluster'],cmap='Set1',alpha=0.65,s=50)
ax.axhline(10,color='white',linestyle='--',alpha=0.4)
ax.set_xlabel('Nombre Absences'); ax.set_ylabel('Moyenne /20')
ax.set_title('Segmentation KMeans — 4 Clusters')
plt.colorbar(sc,label='Cluster'); plt.tight_layout()
plt.savefig('clustering_etudiants.png',dpi=150); plt.show()
print('✅ Clustering terminé — images sauvegardées')

## 8. Détection des Anomalies

In [ ]:
# Outliers par z-score
from scipy import stats
df_notes['zscore'] = df_notes.groupby('module')['note'].transform(lambda x: stats.zscore(x))
anomalies = df_notes[df_notes['zscore'].abs() > 2.5]
print(f'Anomalies détectées (|z| > 2.5) : {len(anomalies)} notes')
print(anomalies[['etudiant_id','module','note','zscore']].sort_values('zscore').head(10))